Note: dropna() (Day 32) rows hata deta hai, lekin fillna() missing values ko kisi reasonable value se replace karta hai — jab data hataana affordable na ho ya important information ho row mein.



1. Data load karo (Day 32 wala cleaned version use karo, jisme Customer ID missing rows already drop ho chuke hain):

In [2]:
import pandas as pd

df = pd.read_csv(
    '../data/processed/step1_customer_id_cleaned.csv',
    dtype={'Invoice': str, 'StockCode': str},
    parse_dates=['InvoiceDate']
)
print(df.shape)
print(df.isnull().sum())

(824364, 8)
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64


2. Simple fillna — ek fixed value se bharna:

In [3]:
# Agar Description missing ho, "Unknown Product" se bharo (drop karne ke bajaye)
df_filled = df.copy()
df_filled['Description'] = df_filled['Description'].fillna('Unknown Product')
print(df_filled['Description'].isnull().sum())   # 0 aana chahiye

0


3.Mean se fillna karna (numerical columns ke liye) — practice ke liye demo:

In [4]:
# NOTE: is dataset mein Price/Quantity mein actually koi NULL nahi hai,
# isliye yahan practice ke liye artificial NaN daal kar dikhate hain
import numpy as np

demo_df = df[['StockCode', 'Price']].copy().head(20)
demo_df.loc[2, 'Price'] = np.nan
demo_df.loc[5, 'Price'] = np.nan
print(demo_df.isnull().sum())

mean_price = demo_df['Price'].mean()
demo_df['Price_filled_mean'] = demo_df['Price'].fillna(mean_price)
print(demo_df)

StockCode    0
Price        2
dtype: int64
   StockCode  Price  Price_filled_mean
0      85048   6.95           6.950000
1     79323P   6.75           6.750000
2     79323W    NaN           3.638333
3      22041   2.10           2.100000
4      21232   1.25           1.250000
5      22064    NaN           3.638333
6      21871   1.25           1.250000
7      21523   5.95           5.950000
8      22350   2.55           2.550000
9      22349   3.75           3.750000
10     22195   1.65           1.650000
11     22353   2.55           2.550000
12    48173C   5.95           5.950000
13     21755   5.45           5.450000
14     21754   5.95           5.950000
15     84879   1.69           1.690000
16     22119   6.95           6.950000
17     22142   1.45           1.450000
18     22296   1.65           1.650000
19     22295   1.65           1.650000


4.Median se fillna karna — kab mean se better hota hai:

In [5]:
median_price = demo_df['Price'].median()
demo_df['Price_filled_median'] = demo_df['Price'].fillna(median_price)
print(demo_df[['Price', 'Price_filled_mean', 'Price_filled_median']])

    Price  Price_filled_mean  Price_filled_median
0    6.95           6.950000                 6.95
1    6.75           6.750000                 6.75
2     NaN           3.638333                 2.55
3    2.10           2.100000                 2.10
4    1.25           1.250000                 1.25
5     NaN           3.638333                 2.55
6    1.25           1.250000                 1.25
7    5.95           5.950000                 5.95
8    2.55           2.550000                 2.55
9    3.75           3.750000                 3.75
10   1.65           1.650000                 1.65
11   2.55           2.550000                 2.55
12   5.95           5.950000                 5.95
13   5.45           5.450000                 5.45
14   5.95           5.950000                 5.95
15   1.69           1.690000                 1.69
16   6.95           6.950000                 6.95
17   1.45           1.450000                 1.45
18   1.65           1.650000                 1.65


Note: Agar column mein outliers hain (jaise kuch bahut mehenge items), to median better hai kyunki mean outliers se skew ho jaata hai. Price column mein tumhare dataset mein bahut variation hai, isliye real scenario mein median zyada safe hota.



5. Group-wise fillna — zyada intelligent approach (bahut useful pattern):

In [6]:
# Har StockCode ka apna average price se us StockCode ki missing prices bharna
# (yeh zyada accurate hai overall mean use karne se, kyunki har product ka apna price range hota hai)
demo_df2 = df[['StockCode', 'Price']].copy()
demo_df2.loc[demo_df2.sample(5, random_state=1).index, 'Price'] = np.nan  # random NaN insert (demo ke liye)

demo_df2['Price'] = demo_df2.groupby('StockCode')['Price'].transform(
    lambda x: x.fillna(x.mean())
)
print(demo_df2.isnull().sum())

StockCode    0
Price        0
dtype: int64


6.Forward-fill aur Backward-fill (time-series jaisa data ke liye — concept samajhna):

In [7]:
# ffill — pichli valid value se bharna
# bfill — agli valid value se bharna
sample_series = pd.Series([100, np.nan, np.nan, 200, np.nan, 300])
print("Original:", sample_series.tolist())
print("Forward fill:", sample_series.ffill().tolist())
print("Backward fill:", sample_series.bfill().tolist())

Original: [100.0, nan, nan, 200.0, nan, 300.0]
Forward fill: [100.0, 100.0, 100.0, 200.0, 200.0, 300.0]
Backward fill: [100.0, 200.0, 200.0, 200.0, 300.0, 300.0]


Real use-case: Agar kisi sensor/stock-price data mein gaps hon, to time ke hisaab se pichli value continue karna sensible hota hai — humare e-commerce dataset mein directly zaroorat nahi, lekin concept jaanna zaroori hai.


7. Apne actual dataset mein decision (README ke liye note):

In [9]:
# Final decision: Description missing -> 'Unknown Product' se fill (drop nahi kiya)
df['Description'] = df['Description'].fillna('Unknown Product')
print(f"Final missing check:\n{df.isnull().sum()}")

df.to_csv('../data/processed/step2_description_filled.csv', index=False)

Final missing check:
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64


Practice questions:

1.Ek chhota demo column banao jisme kuch NaN ho, aur fillna() se mode (sabse common value) se bharne ki koshish karo (series.mode()[0] use karke) — yeh categorical data ke liye best approach hai.

In [12]:
import pandas as pd
import numpy as np

# Ek demo categorical column (jaise Country) kuch NaN values ke sath
demo_country = pd.Series(['UK', 'France', 'UK', np.nan, 'Germany', np.nan, 'UK', 'France'])
print("Original Series with NaNs:\n", demo_country)

# Mode nikalna (sabse zyada baar aane wali value)
most_common_country = demo_country.mode()[0]
print(f"\nMode value: {most_common_country}")

# Fillna use karke missing values ko mode se bharna
demo_country_filled = demo_country.fillna(most_common_country)
print("\nFilled Series:\n", demo_country_filled)

Original Series with NaNs:
 0         UK
1     France
2         UK
3        NaN
4    Germany
5        NaN
6         UK
7     France
dtype: str

Mode value: UK

Filled Series:
 0         UK
1     France
2         UK
3         UK
4    Germany
5         UK
6         UK
7     France
dtype: str


2.Explain karo (comment mein): kab dropna() use karoge aur kab fillna() — apne is project ke context mein dono ke real examples do.


DROPNA vs FILLNA: Kab kya use karein? (Project Context)


1. Kab dropna() use karein?

Ans:- Jab missing value wala column tumhare analysis ke liye "Critical/Mandatory" ho aur uske bina record bekaar ho.
 - Example (Hamare dataset mein): Customer ID. 
   Agar transaction ka Customer ID hi missing hai, toh hum customer-level analysis (jaise Lifetime Value, Repeat Customers) nahi kar sakte. Aise mein row ko hataana (dropna) hi padega kyunki galat ID se data pollute ho jayega.



2. Kab fillna() use karein?

Ans: Jab data drop karna affordable na ho (bohot saara data lose ho jayega) ya missing value ko safely guess/impute kiya ja sake.
 - Example A (Hamare dataset mein - Text/Categorical): Description. 
    Agar kisi product ki description missing hai, toh poori sales transaction delete karna nuksan-deh hai. 

   Isliye humne ise 'Unknown Product' se fill kar diya taaki revenue aur quantity ka data loss na ho.
  - Example B (Numerical): Price ya Quantity. 
    Agar price missing ho, toh hum poori row drop karne ke bajaye median ya group-wise mean price se fill kar sakte hain 
   kyunki product ki pricing pattern se hume sahi andaza mil jata hai.